<a href="https://colab.research.google.com/github/DinethPerera-eng/Statistical-Learning-e22285/blob/main/Assignment_7C_E22285.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 7C: Item Response Prediction and Click-Through Rate Prediction

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.stats import beta as beta_distribution

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Part 1: Bayesian Estimation of User Ability from Item Responses

An online learning platform presents multiple-choice questions one at a time. A response is recorded as

$$
Y_i =
\begin{cases}
1, & \text{if item } i \text{ is answered correctly},\\
0, & \text{if item } i \text{ is answered incorrectly}.
\end{cases}
$$

The response probability is modeled using the two-parameter logistic, or 2PL, item response model:

$$
P(Y_i=1\mid \Theta=\theta)
=
p_i(\theta)
=
\frac{1}{1+\exp[-a_i(\theta-b_i)]}.
$$

Here, $\theta$ is the user's hidden ability, $a_i$ is the discrimination of item $i$, and $b_i$ is its difficulty. Before observing any answers, the ability parameter has the standard normal prior

$$
\Theta\sim N(0,1).
$$

## 1.1 Visualizing the 2PL Response Curves

The probability of answering an item correctly is

$$
p_i(\theta)
=
\frac{1}{1+\exp[-a_i(\theta-b_i)]}.
$$

The difficulty parameter $b_i$ controls the horizontal location of the response curve. At $\theta=b_i$,

$$
p_i(b_i)=\frac{1}{2}.
$$

Therefore, $b_i$ is the ability level at which the user has a 50% chance of answering correctly. Increasing $b_i$ shifts the curve to the right because a greater ability is required to obtain the same success probability.

The discrimination parameter $a_i$ controls the steepness of the curve. A larger value of $a_i$ produces a steeper transition around $b_i$, while a smaller value produces a flatter curve.

In [2]:
theta_values = np.linspace(-4, 4, 700)

curve_settings = [
    {"a": 0.7, "b": 0.0, "label": "a = 0.7, b = 0.0"},
    {"a": 1.7, "b": -1.0, "label": "a = 1.7, b = -1.0"},
    {"a": 1.7, "b": 0.0, "label": "a = 1.7, b = 0.0"},
    {"a": 1.7, "b": 1.0, "label": "a = 1.7, b = 1.0"},
]

fig = go.Figure()

for setting in curve_settings:
    probability = 1 / (
        1 + np.exp(
            -setting["a"] * (theta_values - setting["b"])
        )
    )

    fig.add_trace(
        go.Scatter(
            x=theta_values,
            y=probability,
            mode="lines",
            name=setting["label"],
        )
    )

fig.update_layout(
    title="2PL Probability of a Correct Response",
    xaxis_title="Latent ability, theta",
    yaxis_title="Probability of a correct answer",
    template="plotly_white",
    width=900,
    height=520,
)

fig.update_yaxes(range=[0, 1])
fig.show()

### Interpretation

For $a_i=1.7$, the three curves have the same shape but different horizontal positions. The item with $b_i=-1$ is easier because a relatively low-ability user already has a reasonable chance of answering correctly. The item with $b_i=1$ is more difficult because the curve is shifted to the right and a higher ability is required.

The curve with $a_i=0.7$ is much flatter than the curves with $a_i=1.7$. This means that the easier and harder users are not separated as strongly by the item. A high-discrimination item is more useful for identifying differences between users whose abilities are close to the item's difficulty.

## 1.2 Sequential Likelihood Contribution

For one newly observed response $y_k\in\{0,1\}$, the likelihood contribution is

$$
L(y_k\mid\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k},
$$

where

$$
p_k(\theta)
=
\frac{1}{1+\exp[-a_k(\theta-b_k)]}.
$$

This single expression covers both possible responses:

- When $y_k=1$,

$$
L(y_k\mid\theta)=p_k(\theta).
$$

- When $y_k=0$,

$$
L(y_k\mid\theta)=1-p_k(\theta).
$$

Assuming that responses are conditionally independent given $\theta$, the joint likelihood of the running response history

$$
\mathbf y^{(k)}=(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

The joint likelihood measures how well a possible value of $\theta$ explains all responses observed up to step $k$.

## 1.3 Recursive Bayesian Posterior Update

Let

$$
f_{k-1}(\theta)
=
f_{\Theta\mid\mathbf Y^{(k-1)}}
(\theta\mid\mathbf y^{(k-1)})
$$

be the posterior after the first $k-1$ item responses. When the new response $y_k$ is observed, Bayes' theorem gives

$$
f_k(\theta)
\propto
L(y_k\mid\theta)f_{k-1}(\theta).
$$

Substituting the Bernoulli likelihood gives

$$
f_k(\theta)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{k-1}(\theta).
$$

The normalized posterior is

$$
f_k(\theta)
=
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{k-1}(\theta)
}{
\int_{-\infty}^{\infty}
[p_k(u)]^{y_k}
[1-p_k(u)]^{1-y_k}
f_{k-1}(u)\,du
}.
$$

Initially,

$$
f_0(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

The 2PL likelihood is not conjugate to the normal prior. Therefore, the posterior does not simplify to another normal distribution with simple parameter updates. A numerical method is required to represent and normalize the running posterior.

## 1.4 Dynamic Posterior Shifting after a Difficult Correct Answer

Suppose the user correctly answers a difficult item, so $y_k=1$ and $b_k$ is large. The update becomes

$$
f_k(\theta)\propto p_k(\theta)f_{k-1}(\theta).
$$

For a difficult item, $p_k(\theta)$ is very small at low values of $\theta$. Therefore, multiplying by $p_k(\theta)$ strongly reduces the posterior density assigned to low ability values. Higher values of $\theta$ receive larger likelihood values because high-ability users are more likely to answer the difficult item correctly.

As a result, the posterior distribution usually moves to the right. Its mean and MAP estimate normally increase. The change may be large because a correct response to a difficult item is relatively unlikely for a low-ability user and therefore provides strong evidence of high ability.

In contrast, an incorrect answer to an easy item would downweight high ability values and could shift the posterior to the left.

## 1.5 Effect of the Discrimination Parameter on Certainty

The information supplied by a 2PL item at ability $\theta$ can be expressed as

$$
I_k(\theta)
=
a_k^2p_k(\theta)[1-p_k(\theta)].
$$

This expression shows that information increases with the square of the discrimination parameter.

When $a_k$ is large, the response curve is steep. Near the difficulty value $b_k$, a small difference in ability produces a large difference in the probability of a correct answer. Therefore, the observed response can strongly separate plausible from implausible ability values. The posterior becomes narrower and sharper, and its variance can decrease quickly.

When $a_k$ is small, the response curve is flat. The probability of success changes only slowly across ability values. A correct or incorrect response then provides limited information, so the posterior changes only slightly and remains relatively wide.

The largest information is normally obtained when the item difficulty is close to the user's current estimated ability because $p_k(\theta)[1-p_k(\theta)]$ is largest near $p_k(\theta)=0.5$.

## 1.6 Fixed-Grid Numerical Algorithm

A fixed-grid approximation can be used to maintain the posterior sequentially.

1. Select a wide grid of possible ability values, such as

$$
\theta_j\in[-4,4].
$$

2. Evaluate the standard normal prior on every grid point:

$$
f_0(\theta_j)=\phi(\theta_j).
$$

3. Normalize the grid numerically:

$$
f_0(\theta_j)
\leftarrow
\frac{f_0(\theta_j)}
{\operatorname{trapz}(f_0,\theta)}.
$$

4. For each new item, calculate

$$
p_k(\theta_j)
=
\frac{1}
{1+\exp[-a_k(\theta_j-b_k)]}.
$$

5. Calculate the likelihood at each grid point:

$$
L_k(\theta_j)
=
[p_k(\theta_j)]^{y_k}
[1-p_k(\theta_j)]^{1-y_k}.
$$

6. Multiply the old posterior by the likelihood:

$$
\widetilde f_k(\theta_j)
=
L_k(\theta_j)f_{k-1}(\theta_j).
$$

7. Calculate the numerical normalizing constant:

$$
Z_k
\approx
\operatorname{trapz}
(\widetilde f_k,\theta).
$$

8. Normalize the posterior:

$$
f_k(\theta_j)
=
\frac{\widetilde f_k(\theta_j)}{Z_k}.
$$

9. Calculate the running posterior mean:

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)}
\approx
\operatorname{trapz}
(\theta f_k(\theta),\theta).
$$

10. Calculate the MAP estimate using the grid point with the highest posterior density:

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
=
\theta_{\operatorname{argmax}_j f_k(\theta_j)}.
$$

A fine grid improves accuracy, but it also increases the computational cost.

## 1.7 Complete Simulation for 20 Items

The following script simulates a user with true ability

$$
\theta_{\text{true}}=0.75.
$$

Each item receives a random difficulty

$$
b_k\sim N(0,1)
$$

and a random discrimination

$$
a_k\sim\operatorname{Uniform}(0.5,2.0).
$$

At each step, a response is generated using the true response probability. The posterior is then updated on a fixed grid, and the running posterior mean and MAP are stored.

In [3]:
rng = np.random.default_rng(78)

theta_true = 0.75
number_of_items = 20

theta_grid = np.linspace(-4, 4, 2001)

posterior = norm.pdf(theta_grid, loc=0, scale=1)
posterior = posterior / np.trapezoid(
    posterior,
    theta_grid,
)

initial_mean = np.trapezoid(
    theta_grid * posterior,
    theta_grid,
)
initial_map = theta_grid[np.argmax(posterior)]
initial_variance = np.trapezoid(
    (theta_grid - initial_mean) ** 2 * posterior,
    theta_grid,
)
initial_sd = np.sqrt(initial_variance)

posterior_mean_history = [initial_mean]
posterior_map_history = [initial_map]
posterior_sd_history = [initial_sd]

item_records = []

for step in range(1, number_of_items + 1):

    difficulty_b = rng.normal(0, 1)
    discrimination_a = rng.uniform(0.5, 2.0)

    true_response_probability = 1 / (
        1 + np.exp(
            -discrimination_a
            * (theta_true - difficulty_b)
        )
    )

    response = int(
        rng.random() < true_response_probability
    )

    probability_grid = 1 / (
        1 + np.exp(
            -discrimination_a
            * (theta_grid - difficulty_b)
        )
    )

    if response == 1:
        likelihood = probability_grid
    else:
        likelihood = 1 - probability_grid

    unnormalized_posterior = posterior * likelihood

    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid,
    )

    posterior = (
        unnormalized_posterior
        / normalizing_constant
    )

    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid,
    )

    posterior_map = theta_grid[
        np.argmax(posterior)
    ]

    posterior_variance = np.trapezoid(
        (theta_grid - posterior_mean) ** 2
        * posterior,
        theta_grid,
    )
    posterior_sd = np.sqrt(posterior_variance)

    posterior_mean_history.append(
        posterior_mean
    )
    posterior_map_history.append(
        posterior_map
    )
    posterior_sd_history.append(
        posterior_sd
    )

    item_records.append(
        {
            "Step": step,
            "Difficulty b": difficulty_b,
            "Discrimination a": discrimination_a,
            "True success probability":
                true_response_probability,
            "Observed response": response,
            "Posterior mean": posterior_mean,
            "MAP": posterior_map,
            "Posterior SD": posterior_sd,
        }
    )

item_results = pd.DataFrame(item_records)

steps = np.arange(0, number_of_items + 1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean_history,
        mode="lines+markers",
        name="Posterior mean",
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map_history,
        mode="lines+markers",
        name="MAP estimate",
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability = 0.75",
)

fig.update_layout(
    title="Convergence of Sequential Ability Estimates",
    xaxis_title="Number of answered items",
    yaxis_title="Estimated ability",
    template="plotly_white",
    width=950,
    height=540,
)

fig.show()

print(
    f"Correct responses: "
    f"{item_results['Observed response'].sum()} "
    f"out of {number_of_items}"
)

print(
    f"Final posterior mean: "
    f"{posterior_mean_history[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{posterior_map_history[-1]:.4f}"
)

print(
    f"Initial posterior SD: "
    f"{posterior_sd_history[0]:.4f}"
)

print(
    f"Final posterior SD: "
    f"{posterior_sd_history[-1]:.4f}"
)

display(item_results)

Correct responses: 13 out of 20
Final posterior mean: 0.7489
Final MAP estimate: 0.7360
Initial posterior SD: 0.9995
Final posterior SD: 0.3524


,Step,Difficulty b,Discrimination a,True success probability,Observed response,Posterior mean,MAP,Posterior SD
0,1,-0.676549,1.872703,0.935324,1,0.374171,0.272,0.845824
1,2,-1.093905,1.058502,0.875640,1,0.508461,0.400,0.816475
2,3,0.046412,0.986212,0.666831,1,0.737922,0.644,0.789453
3,4,-0.480352,1.775358,0.898832,1,0.874640,0.756,0.743281
4,5,0.266980,1.565111,0.680480,0,0.421421,0.348,0.611808
5,6,1.011158,1.889071,0.379106,1,0.836570,0.788,0.579101
6,7,0.450988,1.901219,0.638414,1,1.017822,0.956,0.541179
7,8,0.738024,1.250897,0.503745,1,1.156253,1.092,0.530864
8,9,0.052534,1.214156,0.699902,0,0.917773,0.876,0.486983
9,10,-0.622053,0.979700,0.793180,0,0.740797,0.708,0.463582


### Convergence Analysis

At the beginning, the posterior is mainly controlled by the $N(0,1)$ prior. The initial posterior mean and MAP are both close to zero because no item responses have been observed.

After each response, the estimates move upward or downward according to the difficulty, discrimination, and correctness of the item. A correct response to a difficult and highly discriminating item usually causes a strong upward movement. An incorrect response may move the estimates downward, particularly when the item was expected to be easy for a high-ability user.

The estimates do not necessarily move smoothly toward $0.75$. Random responses can produce short-term fluctuations. However, as more items are answered, the combined likelihood becomes more influential than the initial prior. The posterior mean and MAP become more stable and generally move closer to the true ability.

The posterior standard deviation also decreases as information accumulates. This means that the platform is becoming more confident about the user's ability. The final estimate may not equal exactly $0.75$ because only 20 items are used and the responses are random. More well-targeted and highly discriminating items would normally improve the accuracy and reduce uncertainty further.

# Part 2: Bayesian Tracking of Click-Through Rate

An e-commerce platform observes whether users click an advertisement. Each interaction is represented by

$$
Y_k =
\begin{cases}
1, & \text{if the advertisement is clicked},\\
0, & \text{if the advertisement is not clicked}.
\end{cases}
$$

Conditional on the hidden click-through rate $\Theta=\theta$,

$$
Y_k\mid\Theta=\theta
\sim\operatorname{Bernoulli}(\theta).
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(\alpha_0,\beta_0).
$$

The Beta prior is conjugate to the Bernoulli likelihood. Therefore, every posterior remains in the Beta family and can be updated using only two shape parameters.

## 2.1 Structural Properties of the Beta Distribution

The Beta probability density function is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad 0<\theta<1.
$$

Its mean is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

This mean represents the center of mass of the distribution.

- When $\alpha=\beta=1$, the density is uniform and every CTR value is equally likely.
- When $\alpha<\beta$, more probability mass is placed near zero.
- When $\alpha>\beta$, more probability mass is placed near one.
- As $\alpha+\beta$ increases, the distribution becomes more concentrated around its mean.

In [4]:
theta_values = np.linspace(0.001, 0.999, 800)

beta_settings = [
    {
        "alpha": 1,
        "beta": 1,
        "label": "Beta(1, 1): uniform",
    },
    {
        "alpha": 2,
        "beta": 8,
        "label": "Beta(2, 8): right-skewed",
    },
    {
        "alpha": 8,
        "beta": 2,
        "label": "Beta(8, 2): left-skewed",
    },
]

fig = go.Figure()

for setting in beta_settings:
    density = beta_distribution.pdf(
        theta_values,
        setting["alpha"],
        setting["beta"],
    )

    fig.add_trace(
        go.Scatter(
            x=theta_values,
            y=density,
            mode="lines",
            name=setting["label"],
        )
    )

fig.update_layout(
    title="Beta Probability Density Functions",
    xaxis_title="Click-through rate, theta",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=520,
)

fig.show()

### Interpretation

For $\operatorname{Beta}(1,1)$, the mean is $0.5$, but the density is completely flat. This represents maximum initial uncertainty because no CTR value is preferred.

For $\operatorname{Beta}(2,8)$,

$$
E[\Theta]=\frac{2}{10}=0.2.
$$

Most density lies near small CTR values. The distribution is called right-skewed because it has a long tail extending toward larger values.

For $\operatorname{Beta}(8,2)$,

$$
E[\Theta]=\frac{8}{10}=0.8.
$$

Most density lies near large CTR values. Therefore, increasing $\alpha$ relative to $\beta$ shifts the center of mass toward one, while increasing $\beta$ relative to $\alpha$ shifts it toward zero.

## 2.2 Sequential and Joint Likelihoods

For a single interaction,

$$
L(y_k\mid\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

When $y_k=1$, the likelihood is $\theta$. When $y_k=0$, the likelihood is $1-\theta$.

For the complete running history

$$
\mathbf y^{(k)}=(y_1,y_2,\ldots,y_k),
$$

the joint likelihood is

$$
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

Let

$$
s_k=\sum_{i=1}^{k}y_i
$$

be the total number of clicks. Then

$$
L(\mathbf y^{(k)}\mid\theta)
=
\theta^{s_k}(1-\theta)^{k-s_k}.
$$

Therefore, for estimating $\theta$, the full history can be summarized by only two values: the number of clicks and the number of non-clicks.

## 2.3 Beta-Bernoulli Conjugate Posterior Update

Suppose the posterior after step $k-1$ is

$$
f_{k-1}(\theta)
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

After observing $y_k$, Bayes' theorem gives

$$
f_k(\theta)
\propto
L(y_k\mid\theta)f_{k-1}(\theta).
$$

Substituting the likelihood and prior,

$$
f_k(\theta)
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining the powers,

$$
f_k(\theta)
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

Therefore,

$$
\Theta\mid\mathbf Y^{(k)}
\sim
\operatorname{Beta}(\alpha_k,\beta_k),
$$

where

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+1-y_k
}.
$$

After $k$ observations,

$$
\alpha_k
=
\alpha_0+\sum_{i=1}^{k}y_i
$$

and

$$
\beta_k
=
\beta_0+k-\sum_{i=1}^{k}y_i.
$$

The posterior mean is

$$
E[\Theta\mid\mathbf Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

This is a closed-form update because no numerical integration is needed.

## 2.4 Dynamic Shifting of the Beta Posterior

When a click is observed, $y_k=1$. Then

$$
\alpha_k=\alpha_{k-1}+1,
\qquad
\beta_k=\beta_{k-1}.
$$

Increasing $\alpha$ shifts probability mass toward larger values of $\theta$. Therefore, a click increases the running CTR estimate.

When a non-click is observed, $y_k=0$. Then

$$
\alpha_k=\alpha_{k-1},
\qquad
\beta_k=\beta_{k-1}+1.
$$

Increasing $\beta$ shifts probability mass toward smaller values of $\theta$. Therefore, a non-click reduces the running CTR estimate.

When $\alpha_k>1$ and $\beta_k>1$, the posterior mode is

$$
\operatorname{mode}(\Theta\mid\mathbf y^{(k)})
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}.
$$

The Beta-Bernoulli model is computationally efficient because the entire posterior is represented by the two numbers $\alpha_k$ and $\beta_k$. In the non-conjugate 2PL IRT model, the likelihood does not combine with the normal prior to create a standard named distribution. Therefore, a grid or another numerical method is required.

## 2.5 Running Point Estimators

The running posterior mean is

$$
\boxed{
\widehat\theta_{\mathrm{Bayes}}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}.
$$

For $\alpha_k>1$ and $\beta_k>1$, the running MAP estimate is

$$
\boxed{
\widehat\theta_{\mathrm{MAP}}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
}.
$$

Boundary cases should also be considered:

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
=
0
\quad
\text{when }
\alpha_k\le1,\ \beta_k>1,
$$

and

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
=
1
\quad
\text{when }
\alpha_k>1,\ \beta_k\le1.
$$

For the initial uniform prior $\operatorname{Beta}(1,1)$, every value has the same density, so the mode is not unique. In the simulation, $0.5$ is used as a convenient starting value.

## 2.6 Complete CTR Simulation for 100 Impressions

The true hidden CTR is set to

$$
\theta_{\text{true}}=0.35.
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(1,1).
$$

At every step, a random number from $U(0,1)$ is compared with $0.35$. A click occurs when the random number is smaller than the true CTR.

In [5]:
rng = np.random.default_rng(42)

theta_true = 0.35

number_of_impressions = 100

alpha = 1.0
beta_parameter = 1.0

posterior_mean_history = [
    alpha / (alpha + beta_parameter)
]

posterior_map_history = [0.5]

ctr_records = []

for step in range(
    1,
    number_of_impressions + 1,
):

    random_draw = rng.uniform(0, 1)

    if random_draw < theta_true:
        click = 1
    else:
        click = 0

    alpha = alpha + click
    beta_parameter = (
        beta_parameter + 1 - click
    )

    posterior_mean = (
        alpha / (alpha + beta_parameter)
    )

    if alpha > 1 and beta_parameter > 1:
        posterior_map = (
            (alpha - 1)
            / (alpha + beta_parameter - 2)
        )

    elif alpha <= 1 and beta_parameter > 1:
        posterior_map = 0.0

    elif alpha > 1 and beta_parameter <= 1:
        posterior_map = 1.0

    else:
        posterior_map = 0.5

    posterior_mean_history.append(
        posterior_mean
    )
    posterior_map_history.append(
        posterior_map
    )

    ctr_records.append(
        {
            "Step": step,
            "Random draw": random_draw,
            "Click": click,
            "Alpha": alpha,
            "Beta": beta_parameter,
            "Posterior mean": posterior_mean,
            "MAP": posterior_map,
            "Absolute mean error":
                abs(posterior_mean - theta_true),
        }
    )

ctr_results = pd.DataFrame(ctr_records)

steps = np.arange(
    0,
    number_of_impressions + 1,
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean_history,
        mode="lines",
        name="Posterior mean",
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map_history,
        mode="lines",
        name="MAP estimate",
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35",
)

fig.update_layout(
    title="Sequential Bayesian Tracking of Click-Through Rate",
    xaxis_title="Number of impressions",
    yaxis_title="Estimated CTR",
    template="plotly_white",
    width=950,
    height=540,
)

fig.show()

total_clicks = int(
    ctr_results["Click"].sum()
)

print(
    f"Observed clicks: "
    f"{total_clicks} out of "
    f"{number_of_impressions}"
)

print(
    f"Final posterior distribution: "
    f"Beta({alpha:.0f}, {beta_parameter:.0f})"
)

print(
    f"Final posterior mean: "
    f"{posterior_mean_history[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{posterior_map_history[-1]:.4f}"
)

print(
    f"Posterior mean error at step 10: "
    f"{abs(posterior_mean_history[10] - theta_true):.4f}"
)

print(
    f"Posterior mean error at step 100: "
    f"{abs(posterior_mean_history[100] - theta_true):.4f}"
)

display(ctr_results)

Observed clicks: 33 out of 100
Final posterior distribution: Beta(34, 68)
Final posterior mean: 0.3333
Final MAP estimate: 0.3300
Posterior mean error at step 10: 0.1000
Posterior mean error at step 100: 0.0167


,Step,Random draw,Click,Alpha,Beta,Posterior mean,MAP,Absolute mean error
0,1,0.773956,0,1.0,2.0,0.333333,0.000000,0.016667
1,2,0.438878,0,1.0,3.0,0.250000,0.000000,0.100000
2,3,0.858598,0,1.0,4.0,0.200000,0.000000,0.150000
3,4,0.697368,0,1.0,5.0,0.166667,0.000000,0.183333
4,5,0.094177,1,2.0,5.0,0.285714,0.200000,0.064286
...,...,...,...,...,...,...,...,...
95,96,0.630283,0,32.0,66.0,0.326531,0.322917,0.023469
96,97,0.361813,0,32.0,67.0,0.323232,0.319588,0.026768
97,98,0.087650,1,33.0,67.0,0.330000,0.326531,0.020000
98,99,0.118006,1,34.0,67.0,0.336634,0.333333,0.013366


### Performance and Convergence Analysis

At step zero, the posterior mean is $0.5$ because the prior is $\operatorname{Beta}(1,1)$. This value is not close to the true CTR of $0.35$, but it only represents the initial state before observing any advertisement interactions.

During the first few impressions, the running estimates can change sharply. Each click or non-click represents a large proportion of the small dataset. For example, one click among two observations produces a very different estimated CTR from one click among one hundred observations.

As the sample size grows, each new response has a smaller effect. The posterior mean and MAP become more stable because they are based on a larger accumulated history. With the fixed random seed, the simulation observes 33 clicks in 100 impressions. The final posterior is

$$
\operatorname{Beta}(34,68).
$$

The final posterior mean is

$$
\frac{34}{34+68}
=
0.3333,
$$

and the MAP is

$$
\frac{34-1}{34+68-2}
=
0.3300.
$$

Both values are reasonably close to the true CTR of $0.35$. The difference is caused by random sampling variation because the observed proportion of clicks is $33/100=0.33$.

The effect of the initial prior decreases over time. The prior contributes one pseudo-click and one pseudo-non-click, while the dataset eventually contributes 100 actual observations. Therefore, the accumulated evidence becomes much more influential than the original prior. This demonstrates an important property of Bayesian learning: the prior controls early estimates, but sufficient data gradually dominate the posterior.

A stronger prior with larger values of $\alpha_0$ and $\beta_0$ would produce more stable early estimates, but it would also require more data to move the posterior away from the prior belief.